# NB contains code for creating a training dataset for Options + Modeling

In [1]:
!pip install polygon-api-client


     ---------------------------------------- 40.6/40.6 KB 2.0 MB/s eta 0:00:00


You should consider upgrading via the 'C:\Users\yashv\miniconda3\envs\MLProj\python.exe -m pip install --upgrade pip' command.


In [2]:
import requests
import time
import gym
import csv
import pandas as pd
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3 import PPO
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from gym import spaces
from datetime import datetime, timedelta
from pandas.tseries.holiday import USFederalHolidayCalendar
from arch import arch_model
import pytz


import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

Matplotlib is building the font cache; this may take a moment.


## Options Data -- Pull

In [3]:
from polygon import RESTClient
import pandas as pd
client = RESTClient(api_key="r65B9O5aplSJn7BWSo8z8pNH8v2wW5yc")

def make_options_dataframe(contract:str):
    ticker = 'O:'+contract
    # List Quotes
    quotes = []
    quote = client.list_quotes(ticker=ticker, timestamp_gte = '2024-06-13', timestamp_lt='2024-07-29', sort='timestamp')
    for value in quote:
        quotes.append(value)

    print('quotes', len(quotes))
    ask_prices, ask_sizes, bid_prices, bid_sizes, timestamps = [0 for _ in range(len(quotes))], [0 for _ in range(len(quotes))], [0 for _ in range(len(quotes))], [0 for _ in range(len(quotes))], [0 for _ in range(len(quotes))]
    for i in range(len(quotes)):
        ask_prices[i], ask_sizes[i], bid_prices[i], bid_sizes[i], timestamps[i] = quotes[i].ask_price, quotes[i].ask_size, quotes[i].bid_price, quotes[i].bid_size, quotes[i].sip_timestamp
    ask_prices, ask_sizes, bid_prices, bid_sizes, timestamps = ask_prices[::-1], ask_sizes[::-1], bid_prices[::-1], bid_sizes[::-1], timestamps[::-1]

    data = [
        {
            'ask_price': ask_prices,
            'ask_size': ask_sizes,
            'bid_price': bid_prices,
            'bid_size': bid_sizes,
            'sip_timestamp': timestamps
        },
    ]

    df = pd.DataFrame(data)
    df = df.explode(list(df.columns))
    df.reset_index(drop=True, inplace=True)
    
    return df

#     if len(df) > 5:
#         df.to_csv(f'{contract}.csv', index=True)
    

In [4]:
data = make_options_dataframe('AAPL240823C00100000')
data.head()

quotes 65210


,ask_price,ask_size,bid_price,bid_size,sip_timestamp
0,130.3,1,0,0,1720186200075501824
1,0,0,114.3,1,1720186200104215552
2,130.3,1,114.3,1,1720186200104215552
3,130.3,1,0,0,1720186200132803584
4,0,0,112.8,1,1720186200132803584


## Convert Timestamp to ET Timestamp 

In [5]:
data['timestamp'] = pd.to_datetime(data['sip_timestamp'], unit='ns', utc=True)
# Convert UTC to Eastern Time (ET)
data['timestamp'] = data['timestamp'].dt.tz_convert('US/Eastern')
# Remove timezone information and keep as datetime with only date, hour, and minute
data['timestamp'] = data['timestamp'].dt.floor('min').dt.tz_localize(None)

In [6]:
data = data[['timestamp', 'ask_price', 'ask_size', 'bid_price', 'bid_size']]

In [7]:
def resample_to_daily(data):
    data.set_index('timestamp', inplace=True)
    daily_data = data.resample('min').agg({
        'ask_price': 'mean',
        'ask_size': 'sum',
        'bid_price': 'mean',
        'bid_size': 'sum'
    }).dropna()
    return daily_data

df = resample_to_daily(data)

In [8]:
df.head()

,ask_price,ask_size,bid_price,bid_size
timestamp,,,,
2024-07-05 09:30:00,120.892187,1508,116.454687,1487
2024-07-05 09:32:00,123.95,306,121.14375,244
2024-07-05 09:33:00,123.95,317,121.35,311
2024-07-05 09:34:00,123.95,124,121.35,124
2024-07-05 09:35:00,123.95,124,121.35,124


## Pull OHLCV data

In [9]:
import requests

def get_ohlcv_data(ticker, start_date, end_date):
    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/minute/{start_date}/{end_date}"
    params = {
        "apiKey": API_KEY,
        "limit": 50000
    }
    response = requests.get(url, params=params)
    data = response.json()
    if 'results' in data:
        df = pd.DataFrame(data['results'])
        
        df['t'] = pd.to_datetime(df['t'], unit='ms', utc=True)
        df['t'] = df['t'].dt.tz_convert('US/Eastern')
        df['t'] = df['t'].dt.floor('min').dt.tz_localize(None)
        df['timestamp'] = df['t']
        df.set_index('timestamp', inplace=True)
        df = df[['o', 'h', 'l', 'c', 'v']].rename(columns={
            'o': 'Open',
            'h': 'High',
            'l': 'Low',
            'c': 'Close',
            'v': 'Volume',
        })
        df['Adj Close'] = df['Close']  # Assuming Adj Close is the same as Close
        
        return df
    else:
        print(f"No OHLCV data found for {ticker}")
        return pd.DataFrame()
    
    
API_KEY = 'r65B9O5aplSJn7BWSo8z8pNH8v2wW5yc'
ticker = 'AAPL'
start_date = '2024-06-13'
end_date = '2024-07-29'

ohlcv = get_ohlcv_data(ticker, start_date, end_date)
ohlcv.head()

,Open,High,Low,Close,Volume,Adj Close
timestamp,,,,,,
2024-06-13 04:00:00,214.52,215.19,214.22,215.04,9404.0,215.04
2024-06-13 04:01:00,215.04,215.07,214.69,214.80,10099.0,214.80
2024-06-13 04:02:00,214.75,214.92,214.68,214.92,3490.0,214.92
2024-06-13 04:03:00,214.99,215.04,214.93,215.04,8891.0,215.04
2024-06-13 04:04:00,215.04,215.25,215.04,215.09,10331.0,215.09


In [10]:
ohlcv.tail()

,Open,High,Low,Close,Volume,Adj Close
timestamp,,,,,,
2024-07-29 19:55:00,217.0364,217.31,217.0364,217.2200,5385.0,217.2200
2024-07-29 19:56:00,217.2000,217.20,217.2000,217.2000,206.0,217.2000
2024-07-29 19:57:00,217.1900,217.19,217.1900,217.1900,251.0,217.1900
2024-07-29 19:58:00,217.3000,217.65,217.3000,217.6191,431.0,217.6191
2024-07-29 19:59:00,217.6500,217.65,217.1900,217.1900,711.0,217.1900


## Merge data

In [11]:
data_ohlcv_quotes = df.join(ohlcv, how='left', on='timestamp')

In [12]:
data_ohlcv_quotes.head()

,ask_price,ask_size,bid_price,bid_size,Open,High,Low,Close,Volume,Adj Close
timestamp,,,,,,,,,,
2024-07-05 09:30:00,120.892187,1508,116.454687,1487,221.6500,222.1600,221.65,221.9000,1516827.0,221.9000
2024-07-05 09:32:00,123.95,306,121.14375,244,222.2993,222.5000,221.87,221.8850,279368.0,221.8850
2024-07-05 09:33:00,123.95,317,121.35,311,221.8700,222.5600,221.74,222.5326,272206.0,222.5326
2024-07-05 09:34:00,123.95,124,121.35,124,222.5400,222.6300,222.17,222.2600,327508.0,222.2600
2024-07-05 09:35:00,123.95,124,121.35,124,222.2700,222.4273,221.99,222.3300,246878.0,222.3300


In [13]:
data_ohlcv_quotes.rename(columns={
    'Open':'open',
    'High':'high',
    'Low':'low',
    'Close':'close',
    'Volume':'volume',
    'Adj Close':'adj_close'
}, inplace = True)

## Add time to maturity

In [14]:
maturity_date = '240823'
data_ohlcv_quotes['time_to_maturity'] = (pd.to_datetime(f'20{maturity_date}', format='%Y%m%d') - data_ohlcv_quotes.index).days
data_ohlcv_quotes.head()

,ask_price,ask_size,bid_price,bid_size,open,high,low,close,volume,adj_close,time_to_maturity
timestamp,,,,,,,,,,,
2024-07-05 09:30:00,120.892187,1508,116.454687,1487,221.6500,222.1600,221.65,221.9000,1516827.0,221.9000,48
2024-07-05 09:32:00,123.95,306,121.14375,244,222.2993,222.5000,221.87,221.8850,279368.0,221.8850,48
2024-07-05 09:33:00,123.95,317,121.35,311,221.8700,222.5600,221.74,222.5326,272206.0,222.5326,48
2024-07-05 09:34:00,123.95,124,121.35,124,222.5400,222.6300,222.17,222.2600,327508.0,222.2600,48
2024-07-05 09:35:00,123.95,124,121.35,124,222.2700,222.4273,221.99,222.3300,246878.0,222.3300,48


## Adding Greeks and Implied Volatility

In [15]:
from scipy.optimize import minimize
from scipy.stats import norm

# Use black scholes model to back out impllied volatility given option prices

def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2) * T)
    d2 = d1 - sigma * np.sqrt(T)
    return (S* norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2))

def objective_function(sigma, S, K, T, r, option_price):
    return(black_scholes_call(S, K, T, r, sigma) - option_price) **2

def implied_volatility_minimize(option_price, S, K, T, r):
    initial_guess = 0.2
    bounds = [(1e-5, 5.0)]

    result = minimize(
        objective_function, 
        initial_guess,
        args=(S, K, T, r, option_price),
        bounds=bounds,
        method='L-BFGS-B',
        tol=1e-6
    )
    if result.success:
        return(result.x[0])
    else:
        print("failed optimization")
        return( bounds[0][0] if objective_function(bounds[0][0], S, K, T, r, option_price) < objective_function(bounds[0][1], S, K, T, r, option_price) else bounds[0][1])
    

Adding mid price and rfr

In [16]:
#process the data to have the following columns
data_ohlcv_quotes['mid_price'] = (data_ohlcv_quotes['ask_price'] + data_ohlcv_quotes['bid_price'])/2
data_ohlcv_quotes['rfr'] = 0.03909

In [17]:
data_ohlcv_quotes.head()

,ask_price,ask_size,bid_price,bid_size,open,high,low,close,volume,adj_close,time_to_maturity,mid_price,rfr
timestamp,,,,,,,,,,,,,
2024-07-05 09:30:00,120.892187,1508,116.454687,1487,221.6500,222.1600,221.65,221.9000,1516827.0,221.9000,48,118.673437,0.03909
2024-07-05 09:32:00,123.95,306,121.14375,244,222.2993,222.5000,221.87,221.8850,279368.0,221.8850,48,122.546875,0.03909
2024-07-05 09:33:00,123.95,317,121.35,311,221.8700,222.5600,221.74,222.5326,272206.0,222.5326,48,122.65,0.03909
2024-07-05 09:34:00,123.95,124,121.35,124,222.5400,222.6300,222.17,222.2600,327508.0,222.2600,48,122.65,0.03909
2024-07-05 09:35:00,123.95,124,121.35,124,222.2700,222.4273,221.99,222.3300,246878.0,222.3300,48,122.65,0.03909


### Adding Implied Volatility

In [18]:
strike_price = 100

data_ohlcv_quotes['implied_vol'] = data_ohlcv_quotes.apply(
    lambda row: implied_volatility_minimize(
        row['mid_price'], row['close'], strike_price, row['time_to_maturity'], row['rfr']
    ),
    axis=1
)

In [19]:
data_ohlcv_quotes.head()

,ask_price,ask_size,bid_price,bid_size,open,high,low,close,volume,adj_close,time_to_maturity,mid_price,rfr,implied_vol
timestamp,,,,,,,,,,,,,,
2024-07-05 09:30:00,120.892187,1508,116.454687,1487,221.6500,222.1600,221.65,221.9000,1516827.0,221.9000,48,118.673437,0.03909,5.0
2024-07-05 09:32:00,123.95,306,121.14375,244,222.2993,222.5000,221.87,221.8850,279368.0,221.8850,48,122.546875,0.03909,5.0
2024-07-05 09:33:00,123.95,317,121.35,311,221.8700,222.5600,221.74,222.5326,272206.0,222.5326,48,122.65,0.03909,5.0
2024-07-05 09:34:00,123.95,124,121.35,124,222.5400,222.6300,222.17,222.2600,327508.0,222.2600,48,122.65,0.03909,5.0
2024-07-05 09:35:00,123.95,124,121.35,124,222.2700,222.4273,221.99,222.3300,246878.0,222.3300,48,122.65,0.03909,5.0


Set option type to approriate value

In [20]:
# Since this contract is a Call option
data_ohlcv_quotes['option_type'] = 'C'

### Adding Greeks

In [23]:
def calculate_greeks(df, strike_price, r):
    K = strike_price
    df['delta'] = 0
    df['gamma'] = 0
    df['theta'] = 0
    df['vega'] = 0
    df['rho'] = 0
    
    for index, row in df.iterrows():
        S = row['close']
        T = row['time_to_maturity'] / 252
        iv = row['implied_vol']
        
        d1 = (np.log(S/K) + (r + 0.5 * iv**2) * T) / (iv * np.sqrt(T))
        d2 = d1 - iv * np.sqrt(T)
        
        df.at[index,'delta'] = np.where(row['option_type'] == 'C', np.exp(-r * T) * norm.cdf(d1), -np.exp(-r * T) * norm.cdf(-d1))
        df.at[index,'gamma'] = np.exp(-r * T) * norm.pdf(d1) / (S * iv * np.sqrt(T))
        df.at[index,'theta'] = - (S * iv * np.exp(-r * T) * norm.pdf(d1)) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
        df.at[index,'vega'] = S * np.sqrt(T) * np.exp(-r * T) * norm.pdf(d1)
        df.at[index,'rho'] = np.where(row['option_type'] == 'C', K * T * np.exp(-r * T) * norm.cdf(d2), -K * T * np.exp(-r * T) * norm.cdf(-d2))
        
    return df

In [24]:
df = calculate_greeks(data_ohlcv_quotes, strike_price = 100, r=0.05)
df.head()

,ask_price,ask_size,bid_price,bid_size,open,high,low,close,volume,adj_close,time_to_maturity,mid_price,rfr,implied_vol,option_type,delta,gamma,theta,vega,rho
timestamp,,,,,,,,,,,,,,,,,,,,
2024-07-05 09:30:00,120.892187,1508,116.454687,1487,221.6500,222.1600,221.65,221.9000,1516827.0,221.9000,48,118.673437,0.03909,5.0,C,0.919157,0.000281,-173.999277,13.168295,4.439627
2024-07-05 09:32:00,123.95,306,121.14375,244,222.2993,222.5000,221.87,221.8850,279368.0,221.8850,48,122.546875,0.03909,5.0,C,0.919153,0.000281,-173.995367,13.168001,4.439448
2024-07-05 09:33:00,123.95,317,121.35,311,221.8700,222.5600,221.74,222.5326,272206.0,222.5326,48,122.65,0.03909,5.0,C,0.919334,0.000279,-174.163865,13.180684,4.447200
2024-07-05 09:34:00,123.95,124,121.35,124,222.5400,222.6300,222.17,222.2600,327508.0,222.2600,48,122.65,0.03909,5.0,C,0.919258,0.000280,-174.093015,13.175351,4.443939
2024-07-05 09:35:00,123.95,124,121.35,124,222.2700,222.4273,221.99,222.3300,246878.0,222.3300,48,122.65,0.03909,5.0,C,0.919277,0.000280,-174.111219,13.176721,4.444776


In [25]:
df.columns

Index(['ask_price', 'ask_size', 'bid_price', 'bid_size', 'open', 'high', 'low',
       'close', 'volume', 'adj_close', 'time_to_maturity', 'mid_price', 'rfr',
       'implied_vol', 'option_type', 'delta', 'gamma', 'theta', 'vega', 'rho'],
      dtype='object')

## Calculating and Adding Transaction Costs

### Function for TC >

In [27]:
def simulate_adjusted_price_paths(self, S0, T, dt, n_paths):
    N = int(T / dt)
    paths = np.zeros((n_paths, N))

    for i in range(n_paths):
        paths[i, 0] = S0
        for t in range(1, N):
            Z = np.random.standard_normal()
            paths[i, t] = paths[i, t-1] * np.exp((0 - 0.5 * self.sigma**2) * dt + self.sigma * np.sqrt(dt) * Z)

    return paths

def calculate_transaction_costs(self, paths, eta, phi, k):
    transaction_costs = np.zeros(paths.shape[0])
    for i in range(paths.shape[0]):
        for t in range(1, paths.shape[1]):
            rho = paths[i, t] - paths[i, t-1]
            execution_cost = eta * np.abs(rho)**(1 + phi)
            market_impact = k * np.abs(rho)
            transaction_costs[i] += execution_cost + market_impact
    return transaction_costs

In [26]:
def add_TC(data):
    
    def simulate_adjusted_price_paths(S0, T, dt, n_paths, iv):
        N = int(T / dt)
        paths = np.zeros((n_paths, N))

        for i in range(n_paths):
            paths[i, 0] = S0
            for t in range(1, N):
                Z = np.random.standard_normal()
                paths[i, t] = paths[i, t-1] * np.exp((0 - 0.5 * iv**2) * dt + iv * np.sqrt(dt) * Z)

        return paths

    def calculate_transaction_costs(paths, eta, phi, k):
        transaction_costs = np.zeros(paths.shape[0])
        for i in range(paths.shape[0]):
            for t in range(1, paths.shape[1]):
                rho = paths[i, t] - paths[i, t-1]
                execution_cost = eta * np.abs(rho)**(1 + phi)
                market_impact = k * np.abs(rho)
                transaction_costs[i] += execution_cost + market_impact
        average_transaction_cost = sum(transaction_costs) / paths.shape[0]
        return average_transaction_cost
    
    data['transaction_cost'] = 0
    
    for idx, row in data.iterrows():
        iv = data['implied_vol'].loc[idx]
        paths = simulate_adjusted_price_paths(row['close'], row['time_to_maturity']/365, 1/252, 1000, iv)
        data.at[idx, 'transaction_cost'] = calculate_transaction_costs(paths, 0.01, 0.852, 0.001)
        
    return data

df = add_TC(df)

df.head()

,ask_price,ask_size,bid_price,bid_size,open,high,low,close,volume,adj_close,...,mid_price,rfr,implied_vol,option_type,delta,gamma,theta,vega,rho,transaction_cost
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-07-05 09:30:00,120.892187,1508,116.454687,1487,221.6500,222.1600,221.65,221.9000,1516827.0,221.9000,...,118.673437,0.03909,5.0,C,0.919157,0.000281,-173.999277,13.168295,4.439627,3100.820825
2024-07-05 09:32:00,123.95,306,121.14375,244,222.2993,222.5000,221.87,221.8850,279368.0,221.8850,...,122.546875,0.03909,5.0,C,0.919153,0.000281,-173.995367,13.168001,4.439448,3309.534914
2024-07-05 09:33:00,123.95,317,121.35,311,221.8700,222.5600,221.74,222.5326,272206.0,222.5326,...,122.65,0.03909,5.0,C,0.919334,0.000279,-174.163865,13.180684,4.447200,3875.371707
2024-07-05 09:34:00,123.95,124,121.35,124,222.5400,222.6300,222.17,222.2600,327508.0,222.2600,...,122.65,0.03909,5.0,C,0.919258,0.000280,-174.093015,13.175351,4.443939,1945.341564
2024-07-05 09:35:00,123.95,124,121.35,124,222.2700,222.4273,221.99,222.3300,246878.0,222.3300,...,122.65,0.03909,5.0,C,0.919277,0.000280,-174.111219,13.176721,4.444776,2664.376602


## ML Modeling

In [28]:
df_new = df.copy()

### Adding forecasts to data

In [29]:
from joblib import Parallel, delayed
from statsmodels.tsa.api import ARIMA, ExponentialSmoothing
from tqdm import tqdm  # Import tqdm for progress tracking
import time

def forecast_row(idx, data, forecast_steps, columns, window_size):
    row_forecasts = {}
    row_forecasts['timestamp'] = data.index[idx]

    for indicator, column in columns.items():
        for key, (steps, freq) in forecast_steps.items():
            if indicator not in key:
                continue
            try:
                # Use a sliding window
                start_idx = max(0, idx - window_size)
                series = data[column].iloc[start_idx:idx+1]
                if len(series) < 2:
                    row_forecasts[f'forecast_{indicator}_{key}'] = None
                    continue

                if indicator in ['open', 'high', 'low', 'close', 'transaction_cost', 'delta','gamma', 'theta','vega',
                                'rho','implied_vol','time_to_maturity','mid_price','rfr']:
                    model = ExponentialSmoothing(series, trend='add', seasonal=None)
#                 elif indicator == 'volatility':
#                     model = ARIMA(series, order=(5, 1, 0))
#                 elif indicator == 'volume':
#                     shift = 1 if series.min() <= 0 else 0
#                     transformed_series = np.log(series + shift + 1)
#                     model = ExponentialSmoothing(transformed_series, trend='add', seasonal=None)
                model_fit = model.fit()

                forecast_values = model_fit.forecast(steps=steps)
                row_forecasts[f'forecast_3Hr_{indicator}'] = forecast_values.iloc[-1]

            except Exception as e:
                row_forecasts[f'forecast_3Hr_{indicator}'] = None

    return row_forecasts

def train_and_forecast_parallel(data, forecast_steps, window_size, n_jobs=-1):
    columns = {
        'open': 'open',
        'high': 'high',
        'low': 'low',
        'close': 'close',
        'transaction_cost': 'transaction_cost',
        'delta': 'delta',
        'gamma': 'gamma',
        'theta': 'theta',
        'vega': 'vega',
        'rho': 'rho',
        'implied_vol': 'implied_vol',
        'time_to_maturity': 'time_to_maturity',
        'mid_price': 'mid_price',
        'rfr': 'rfr'
    }
    
    print('Starting...')
    # Add tqdm progress bar
    results = Parallel(n_jobs=n_jobs)(delayed(forecast_row)(idx, data, forecast_steps, columns, window_size) for idx in tqdm(range(len(data)), desc="Processing rows"))

    forecast_results = pd.DataFrame(results).set_index('timestamp')
    return forecast_results

# forecast steps -- 3 hours
forecast_steps = {
    'open': (180, '1T'),
    'high': (180, '1T'),
    'low': (180, '1T'),
    'close': (180, '1T'),
    'transaction_cost': (180, '1T'),
    'delta': (180, '1T'),
    'gamma': (180, '1T'),
    'theta': (180, '1T'),
    'vega': (180, '1T'),
    'rho': (180, '1T'),
    'implied_vol': (180, '1T'),
    'time_to_maturity': (180, '1T'),
    'mid_price': (180, '1T'),
    'rfr': (180, '1T')
}

df_new.index = pd.to_datetime(df_new.index)
df_new = df_new.asfreq('T')

# sliding window size (last 100 observations)
window_size = 100

start_time = time.time()

forecasted_data = train_and_forecast_parallel(df_new, forecast_steps, window_size)

end_time = time.time()
elapsed_time = end_time - start_time

# # Fill/drop NaN values
# forecasted_data = forecasted_data.fillna(method='ffill').fillna(method='bfill')

# Combine the forecasted data
train_data = df.join(forecasted_data)
print(train_data.head())
print(f"Elapsed time: {elapsed_time:.2f} seconds")

Starting...



Processing rows:   0%|          | 16/30630 [00:01<39:50, 12.81it/s]/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:918: Conver

KeyboardInterrupt: 

## Dynamic Forecasts

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

def dynamic_forecast(data, step):
    """
    Generates a dynamic forecast for a set of indicators using historical data.

    The function forecasts various financial indicators such as OHLC (Open, High, Low, Close),
    volume, volatility, and technical indicators like RSI, MACD, etc., based on the past data.

    The forecasts are generated using either Exponential Smoothing or ARIMA models, depending on the indicator.

    Args:
    - data (pd.DataFrame): The historical market data.
    - step (int): The forecast step, indicating how far into the future the prediction should be made.

    Returns:
    - pd.DataFrame: A DataFrame containing the forecasted values for the last row in the dataset.
    """
    
    def forecast_last_row(data, forecast_steps, columns, window_size):
        """
        Forecasts the last row of data based on the given columns and their forecast steps.

        Args:
        - data (pd.DataFrame): The historical market data.
        - forecast_steps (dict): A dictionary specifying the forecast steps and frequency for each indicator.
        - columns (dict): A dictionary mapping each indicator to its corresponding column name in the data.
        - window_size (int): The number of past observations to consider for forecasting.

        Returns:
        - tuple: Two DataFrames, one for the row-level forecast and another for the state forecast.
        """
        last_idx = len(data) - 1
        row_forecasts = {'timestamp': data.index[last_idx]}
        state_forecasts = {}
        
        for indicator, column in columns.items():
            steps, freq = forecast_steps[indicator]
            start_idx = max(0, last_idx - window_size)
            series = data[column].iloc[start_idx:last_idx+1]
            
            # If insufficient data, skip forecast for this indicator
            if len(series) < 2:
                row_forecasts[f'forecast_{indicator}'] = None
                continue
            
            # Select appropriate model based on the indicator
            if indicator in ['open', 'high', 'low', 'close', 'RSI', 'MACD',
                           'MACD_signal', 'MACD_hist', 'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB',
                           'Middle_BB', 'Lower_BB', 'ATR_1', 'ATR_2', 'ATR_5', 'ATR_10', 'ATR_20',
                           'ADX', '+DI', '-DI', 'CCI', 'volatility','5_min_volatility', '5_min_volume']:
                model = ExponentialSmoothing(series, trend='add', seasonal=None)
            elif indicator == '5_min_TC' or indicator == 'transaction_cost':
                model = ARIMA(series, order=(5, 1, 0))
            elif indicator == 'volume' or indicator == '5_min_volume':
                shift = 1 if series.min() <= 0 else 0
                transformed_series = np.log(series + shift + 1)
                model = ExponentialSmoothing(transformed_series, trend='add', seasonal=None)
            
            # Fit the model and generate forecast
            model_fit = model.fit()
            forecast_values = model_fit.forecast(steps=steps)
            
            if indicator == 'volume':
                forecast_values = np.exp(forecast_values) - 1 - shift
                forecast_values[forecast_values < 0] = 0  # Ensure non-negative values
                
            # Add forecast to state_forecast (for specific future step)
            state_forecasts[f'{indicator}'] = forecast_values.iloc[step-1]
            
            # Add future OHLCV to row_forecasts (e.g., forecast for the last step in 6 hours)
            if indicator in ['open', 'high', 'low', 'close', 'transaction_cost', 'volume', 'volatility']:
                row_forecasts[f'forecast_6Hr_{indicator}'] = forecast_values.iloc[-1]
        
        # Create DataFrames for row-level and state forecasts
        row_forecasts_df = pd.DataFrame([row_forecasts]).reset_index(drop=True)
        state_forecasts_df = pd.DataFrame([state_forecasts]).reset_index(drop=True)
        
        return row_forecasts_df, state_forecasts_df

    # Define forecast steps and frequency for various indicators
    forecast_steps = {
        'open': (step + 360, '1T'),
        'high': (step + 360, '1T'),
        'low': (step + 360, '1T'),
        'close': (step + 360, '1T'),
        'volume': (step + 360, '1T'),
        'volatility': (step + 360, '1T'),
        'transaction_cost': (step + 360, '1T'),
        'RSI': (step, '1T'),
        'MACD': (step, '1T'),
        'MACD_signal': (step, '1T'),
        'MACD_hist': (step, '1T'),
        'Stoch_k': (step, '1T'),
        'Stoch_d': (step, '1T'),
        'OBV': (step, '1T'),
        'Upper_BB': (step, '1T'),
        'Middle_BB': (step, '1T'),
        'Lower_BB': (step, '1T'),
        'ATR_1': (step, '1T'),
        'ADX': (step, '1T'),
        '+DI': (step, '1T'),
        '-DI': (step, '1T'),
        'CCI': (step, '1T'),
        '5_min_volatility': (step, '1T'),
        '5_min_volume': (step, '1T'),
        '5_min_TC': (step, '1T') # Add bid ask forecasts as well later
    }
    
    # Map indicators to corresponding columns in the data
    columns = {col: col for col in forecast_steps}
    
    # Forecast the last row and state for the given data
    last_row_forecast, data_forecast = forecast_last_row(data, forecast_steps, columns, 300)
    
    # Combine the two DataFrames into a single input row
    input_row = pd.concat([data_forecast, last_row_forecast], axis=1)
    
    return input_row

# Usage
result = dynamic_forecast(data_testing, 360)
result.head()

In [24]:
# Calculate the split index
split_index = int(0.8 * len(data_ohlcv_quotes))

# Split the data into training and testing sets
data_training = data_ohlcv_quotes.iloc[:split_index]
data_testing = data_ohlcv_quotes.iloc[split_index:]

### Training Environment:

In [ ]:
import gym
from gym import spaces
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from datetime import datetime, timedelta

class TradingEnvironment(gym.Env):
    metadata = {'render.modes': ['human']}
    
    # Preferred timeframe: The number of steps that the user wants to complete the trade in
    def __init__(self, data, action_space, preferred_timeframe=390, initial_inventory=10, scenario='medium'):
        super(TradingEnvironment, self).__init__()
        self.data = data
        self.current_step = 0
        
        self.preferred_timeframe = preferred_timeframe
        self.initial_inventory = initial_inventory
        self.remaining_inventory = self.initial_inventory
        self.elapsed_time = 0
        self.trades = []
        self.cumulative_reward = 0
        self.scenario = scenario
        
        # Define scenario-specific penalties
        if self.scenario == 'large':
            self.beta = 1
            self.delta = 1
        elif self.scenario == 'medium-large':
            self.beta = 1e2
            self.delta = 1e2
        elif self.scenario == 'medium':
            self.beta = 1e3
            self.delta = 1e3
        elif self.scenario == 'small-medium':
            self.beta = 1e4
            self.delta = 1e4
        elif self.scenario == 'small':
            self.beta = 1e5
            self.delta = 1e5
        else:
            raise ValueError(f"Unknown scenario: {self.scenario}")

        # Extract state columns - have to change according to requirement
        self.state_columns = ['ask_price', 'ask_size', 'bid_price', 'bid_size', 'open', 'high', 'low',
                               'close', 'volume', 'adj_close', 'time_to_maturity', 'mid_price', 'rfr',
                               'implied_vol', 'option_type', 'delta', 'gamma', 'theta', 'vega', 'rho', 'transaction_cost']
        
        # Define action space - [%_slice, timing_of_next_slice] * Sequence_Length
        self.action_space = action_space
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(len(self.state_columns),), dtype=np.float32)
        
    
    def _get_state(self):
        market_conditions = self._next_observation()
#         state = np.append(market_conditions, self.remaining_inventory / self.initial_inventory)
        return market_conditions
    
    def _next_observation(self):
        return self.data[self.state_columns].iloc[self.current_step].values
    
    def reset(self):
        # print('------------------------------------------------Class resetted------------------------------------------------')
        self.current_step = 0
        self.cumulative_reward = 0
        self.remaining_inventory = self.initial_inventory
        self.elapsed_time = 0
        self.trades = []
        return self._get_state()
    
    def step(self, action):
        # Adding some noise to the actions
        action = self._add_noise_to_action(action)
        # print(f'Action taken: {action}')

        size_of_slice = action[0] * self.remaining_inventory
        # Convert size of slice into whole number
        size_of_slice = int(np.ceil(size_of_slice))

        # Scale action[1] to a desired range, e.g., 1 to 10
        timing_of_slice = int(np.ceil(action[1]))
        # print(f'timing_of_slice: {timing_of_slice}')
        
        self.elapsed_time += timing_of_slice
        
        if self.elapsed_time >= self.preferred_timeframe:
            size_of_slice = self.remaining_inventory

        execution_price = self._take_action(size_of_slice)

        # Print current and next step for debugging
        # print(f'Current step: {self.current_step}')
        self.current_step += timing_of_slice
        self.elapsed_time += timing_of_slice
        # print(f'Next step: {self.current_step}')

        # Ensure the index is sequential
        if self.current_step >= len(self.data):
            self.current_step = len(self.data) - 1

        done = self.remaining_inventory <= 0 or self.elapsed_time >= self.preferred_timeframe
        reward = self._calculate_reward(size_of_slice, execution_price, timing_of_slice)
        self.cumulative_reward += reward
        # if done:
        #     print(f'Cumulative Rewards: {self.cumulative_reward}')

        trade_info = {
                'step': self.current_step,
#                 'timestamp': self.data.index[self.current_step],
                'action': action,
                'price': execution_price,
                'options': size_of_slice,
                'reward': reward,
                'inventory': self.remaining_inventory,
                'time left': self.preferred_timeframe - self.elapsed_time
            }
        self.trades.append(trade_info)

        info = {
            'step': self.current_step,
            'action': action,
            'price': execution_price
        }

        return self._get_state(), reward, done, info

            
    def _add_noise_to_action(self, action):
        # Add noise to the first action (percentage of inventory)
        noise_action_0 = np.random.normal(0, 0.02, size=action[0].shape)  # Small noise for percentage
        action[0] += noise_action_0
        action[0] = np.clip(action[0], self.action_space.low[0], self.action_space.high[0])

        # Add noise to the second action (timing of next slice)
        noise_action_1 = np.random.normal(0, 5, size=action[1].shape)  # Setting SD to be 5% of the range (1-100)
        action[1] += noise_action_1
        action[1] = np.clip(action[1], self.action_space.low[1], self.action_space.high[1])

        return action

    
    def _take_action(self, size_of_slice):
        self.remaining_inventory -= size_of_slice
        print(f'Remaining inventory: {self.remaining_inventory}')
        if self.remaining_inventory < 0:
            self.remaining_inventory = 0
        execution_price = self.data['close'].iloc[self.current_step]
        return execution_price
    
#     def _calculate_transaction_cost(self, volume, daily_volume, volatility=None):
#         if volatility is None:
#             volatility = self.data['volatility'].iloc[self.current_step]
#         return volatility * np.sqrt(volume / daily_volume)
            
    def _calculate_reward(self, size_of_slice, execution_price, timing_of_slice):
        # Constants
        kappa = 0.1

        expected_price = self.data['expected_price'].iloc[self.current_step]
        actual_price = execution_price
        order_size = size_of_slice
        market_liquidity = self.data['market_liquidity'].iloc[self.current_step]
        time_remaining = self.preferred_timeframe - self.elapsed_time
        total_time = self.preferred_timeframe

        # Calculating various components of the reward function
        slippage = expected_price - actual_price
        transaction_costs = self.data['transaction_cost'].iloc[self.current_step]
        
        # Penalize actions taken early in the timeframe (encourage spreading actions)
        early_action_penalty = self.delta * (time_remaining / total_time) ** 2  # Quadratic scaling
        
        if self.scenario in ['small', 'small-medium']:
            small_timestep_penalty = 0 if timing_of_slice > 40 else 100
        elif self.scenario in ['medium', 'medium-large']:
            small_timestep_penalty = 0 if timing_of_slice > 20 else 50
        elif self.scenario == 'large':
            small_timestep_penalty = 0 if timing_of_slice > 10 else 10
        else:
            small_timestep_penalty = 0

        # Combining all the components to calculate the reward
        penalty = (slippage + transaction_costs + early_action_penalty + small_timestep_penalty)
        
        # Adding utility theory in rewards
        reward = -penalty - (2 * kappa * (penalty ** 2))
        
        # print(f"Slippage: {slippage} TC: {transaction_costs} Rapid: {early_action_penalty} Time: {small_timestep_penalty}")

        return reward

    
    def render(self, mode='human', close=False):
        print('--------------------------------------------------')
        print(f'Steps: {self.current_step}')
        print(f'Remaining inventory: {self.remaining_inventory}')
        print(f'Cumulative reward: {self.cumulative_reward}')
        self.print_trades()

    def print_trades(self):
        trades_df = pd.DataFrame(self.trades)
        for trade in self.trades:
            print(f"Step: {trade['step']}, Action: {trade['action']}, Price: {trade['price']}, Shares: {trade['shares']}, Reward: {trade['reward']}, Inventory: {trade['inventory']}, TimeLeft: {trade['time left']}")
        
        return self.trades

## Training Loop